<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# EarthDaily Agriculture - Management Zones (SAMZ) Extraction

Development notebook for the `ZoningExtractor` class.
Tests all four processing modes: **stats**, **stats_geo**, **links**, and **file**.

## Step 1: Initialisation

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## Step 2: Get entities

### Option 1 - Load entities from EarthDaily platform

In [ ]:
manager.load_seasonfields()

print("First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 2 - Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path)
print(f"Loaded {len(manager.sfd_list)} entities from {file_path}")
print(manager.sfd_list.columns.tolist())

### Option 3 - Built-in test dataset (no external dependency)

Small in-memory set of 4 fields in southern Kansas (same area as `test_001` below, which already
returns valid Sentinel-2 coverage). Use this when you don't want to hit the platform or load a file.

**Required columns for the zoning workflow:**
- `id` — entity identifier (string)
- `name` — display name
- `geometry` — WKT polygon

`image_id` is **not** required here — it is added per entity by `CoverageExtractor` in Step 5
before zoning runs.

In [ ]:
import pandas as pd

# 4 ~20-30ha fields around Anthony / Harper County, KS — Sentinel-2 tile 14SLG
test_fields = pd.DataFrame([
    {
        "id": "test_zoning_001",
        "name": "Kansas_Field_A",
        "geometry": (
            "POLYGON ((-97.70066562 37.14062335, -97.69927729 37.14227539, "
            "-97.69935777 37.14233954, -97.70004188 37.14248389, "
            "-97.70008212 37.14359058, -97.69969534 37.14450741, "
            "-97.70262024 37.14450741, -97.70254625 37.14062335, "
            "-97.70066562 37.14062335))"
        ),
    },
    {
        "id": "test_zoning_002",
        "name": "Kansas_Field_B",
        "geometry": (
            "POLYGON ((-97.69500 37.14500, -97.69000 37.14500, "
            "-97.69000 37.14900, -97.69500 37.14900, -97.69500 37.14500))"
        ),
    },
    {
        "id": "test_zoning_003",
        "name": "Kansas_Field_C",
        "geometry": (
            "POLYGON ((-97.71200 37.13500, -97.70700 37.13500, "
            "-97.70700 37.13900, -97.71200 37.13900, -97.71200 37.13500))"
        ),
    },
    {
        "id": "test_zoning_004",
        "name": "Kansas_Field_D",
        "geometry": (
            "POLYGON ((-97.68500 37.13800, -97.68000 37.13800, "
            "-97.68000 37.14200, -97.68500 37.14200, -97.68500 37.13800))"
        ),
    },
])

manager.sfd_list = test_fields
print(f"🧪 Built-in test dataset: {len(manager.sfd_list)} entities")
display(manager.sfd_list)

## Step 3: Get coverage (image IDs)

SAMZ requires an `image_id` per entity. Use CoverageExtractor to find available images first.

In [ ]:
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor
cov_extractor = CoverageExtractor(manager.bearer_token, manager.token_expiration, config=manager.config)

cov_extractor.setup_coverage_parameters(
    vegetation_index='NDVI',
    start_date='2025-01-01',
    end_date='2025-08-01',
    clear_cover_min=95,
)

In [ ]:
# Test coverage on a single entity
test_entity = {
    "id": "test_001",
    "geometry": "POLYGON ((-97.70066562 37.14062335, -97.69927729 37.14227539, -97.69935777 37.14233954, -97.70004188 37.14248389, -97.70008212 37.14359058, -97.69969534 37.14450741, -97.70262024 37.14450741, -97.70254625 37.14062335, -97.70066562 37.14062335))"
}

cov_result = cov_extractor.get_satellite_coverage_by_geometry(test_entity)
cov_df = cov_extractor.format_coverage_json(cov_result)
print(f"Found {len(cov_df)} images")

# Pick the best image and inject into entity
test_entity["image_id"] = cov_df["image_id"].iloc[:3].tolist()
print(test_entity)

## Step 4: Extract management zones

Test ZoningExtractor with all three processing modes.

### Configure extraction (stats mode)

In [ ]:
from earthdaily.agriculture.extractors.zoning_functions import ZoningExtractor
zoning_extractor = ZoningExtractor(manager.bearer_token, manager.token_expiration, config=manager.config)

zoning_extractor.setup_zoning_parameters(
    num_zones=5,
    postprocess='stats',
    output_epsg=4326,
)

### Test API call

In [ ]:
print("--- Test: get_zoning_map ---")
try:
    raw_response = zoning_extractor.get_zoning_map_api(test_entity)
    print("Raw API response received")
    print(type(raw_response))
except Exception as e:
    print(f"Error: {e}")

### Test safe API call

In [ ]:
print("--- Test: get_zoning_map_safe ---")
safe_result = zoning_extractor.get_zoning_map_api_safe(test_entity)
print(f"Success: {safe_result['success']}")
print(f"Error: {safe_result['error']}")

### Test format_zoning_stats_json

In [ ]:
print("--- Test: format_zoning_stats_json ---")
if safe_result['success'] and safe_result['data']:
    stats_df = zoning_extractor.format_zoning_stats_json(safe_result['data'])
    print(f"Stats DataFrame: {stats_df.shape}")
    display(stats_df)
else:
    print("No data to format")

### Test process_single_entity_zoning (stats)

In [ ]:
import pandas as pd

row = pd.Series({
    "id": "test_001",
    "name": "Test_Field",
    "geometry": test_entity["geometry"],
    "image_id": test_entity["image_id"],
})

result = zoning_extractor.process_single_entity_zoning(row)
print(f"Error: {result['error']}")
if result['data'] is not None:
    print(f"Data shape: {result['data'].shape}")
    display(result['data'])

### Test links mode

In [ ]:
zoning_extractor.setup_zoning_parameters(
    num_zones=5,
    postprocess='links',
    directLinks=True,
    output_epsg=4326,
)

result_links = zoning_extractor.process_single_entity_zoning(row)
print(f"Error: {result_links['error']}")
if result_links['data'] is not None:
    print(f"Data shape: {result_links['data'].shape}")
    display(result_links['data'])

### Test file mode (PNG)

In [ ]:
zoning_extractor.setup_zoning_parameters(
    num_zones=5,
    postprocess='file',
    map_format='png',
    output_path=manager.output_result_dir,
    output_epsg=4326,
)

result_file = zoning_extractor.process_single_entity_zoning(row)
print(f"Error: {result_file['error']}")
if result_file['data'] is not None:
    print(f"Data shape: {result_file['data'].shape}")
    display(result_file['data'])

### Test stats_geo mode (stats + zone geometry)

In [ ]:
zoning_extractor.setup_zoning_parameters(
    num_zones=5,
    postprocess="stats_geo",
    output_epsg=4326,
)

result_stats_geo = zoning_extractor.process_single_entity_zoning(row)
print(f"Error: {result_stats_geo['error']}")
if result_stats_geo["data"] is not None:
    print(f"Data shape: {result_stats_geo['data'].shape}")
    print(f"Columns: {result_stats_geo['data'].columns.tolist()}")
    display(result_stats_geo["data"])
    # Show zone geometries (first 100 chars each)
    if "zone_geometry" in result_stats_geo["data"].columns:
        for _, r in result_stats_geo["data"].iterrows():
            geom = str(r.get("zone_geometry", ""))[:100]
            print(f"  Zone {r['zone_name']}: PI={r['productivity_index']}, VI={r['variability_index']}, geom={geom}...")

## Step 5: Bulk extraction

Run zoning on multiple entities in parallel.

In [ ]:
# First get coverage for all entities to obtain image_ids
cov_results = cov_extractor.process_entity_coverage_bulk_parallel(
    entity_list=manager.sfd_list.head(10),
    max_workers=5,
    skip_export=True,
)

cov_df = cov_results["results_df"]
print(f"Coverage results: {len(cov_df)} rows")

# Aggregate: pick top 3 image_ids per entity (sorted by date desc / coverage)
image_ids_per_entity = (
    cov_df.groupby("id")["image_id"]
    .apply(lambda x: x.head(3).tolist())
    .reset_index()
)

# Merge image_id lists back onto original entity data (id, geometry, etc.)
entities = manager.sfd_list.head(10).copy()
zoning_input = entities.merge(
    image_ids_per_entity,
    on="id",
    how="inner"
)

print(f"Zoning input: {len(zoning_input)} entities with image_id lists")
display(zoning_input[["id", "image_id"]].head())

In [ ]:
# Setup zoning in stats mode
zoning_extractor.setup_zoning_parameters(
    num_zones=5,
    postprocess='stats',
    output_epsg=4326,
)

zoning_results = zoning_extractor.process_entity_zoning_bulk_parallel(
    entity_list=zoning_input,
    max_workers=5,
    output_path=manager.output_result_dir,
    skip_export=False,
    prefix='zoning',
)

print(f"Total: {zoning_results['total_calculations']}")
print(f"Successful: {zoning_results['successful_calculations']}")
print(f"Failed: {zoning_results['failed_calculations']}")
if not zoning_results['results_df'].empty:
    display(zoning_results['results_df'].head())

In [ ]:
zoning_results['results_df'].dtypes
type(zoning_results['results_df']['image_id'].iloc[0]) 

## Step 6: Workflow Integration Test

Test the ZoningExtractor in the pattern used by `WorkflowManager.run_workflow()`.
In the fertilizer zoning pipeline, upstream CropID results are transformed into
an entity list with `image_id` as a **list of strings** per entity (top N images per crop year).

**YAML config reference** (`configuration/crop_history_zoning.yml`):
```yaml
steps:
  - name: zoning
    depends_on: smart_crop_coverage
    extractor: ZoningExtractor
    module: earthdaily.agriculture.extractors.zoning_functions
    setup: { method: setup_zoning_parameters, params: { num_zones: 5, postprocess: stats } }
    run: { method: process_entity_zoning_bulk_parallel }
```

In [ ]:
# Workflow-style dynamic import & setup (mirrors WorkflowManager._execute_single_step)
import importlib

module_path = "earthdaily.agriculture.extractors.zoning_functions"
class_name = "ZoningExtractor"

mod = importlib.import_module(module_path)
ExtractorClass = getattr(mod, class_name)

wf_zoning = ExtractorClass(manager.bearer_token, manager.token_expiration, config=manager.config)

# Setup with workflow parameters
wf_zoning.setup_zoning_parameters(
    num_zones=5,
    postprocess="stats",
)

print("Workflow-style ZoningExtractor ready")

### Test image_id as list (workflow transform output)

The upstream `smart_crop_coverage` transform produces `image_id` as a list of image ID strings.
ZoningExtractor must accept both `str` and `list[str]` for backward compatibility.

In [ ]:
# Simulate workflow transform output: image_id as list
import pandas as pd

# Build test entity with image_id as a list (as produced by smart_crop_coverage transform)
test_row = pd.Series({
    "id": manager.sfd_list.iloc[0]["id"],
    "geometry": manager.sfd_list.iloc[0]["geometry"],
    "image_id": cov_df.groupby("id")["image_id"].apply(list).iloc[0][:3],  # top 3 images as list
})
print(f"Entity: {test_row['id']}")
print(f"image_id type: {type(test_row['image_id'])} = {test_row['image_id']}")

# Process single entity with list image_id
result = wf_zoning.process_single_entity_zoning(test_row)

if result["data"] is not None:
    print(f"Result columns: {list(result['data'].columns)}")
    display(result["data"].head())
else:
    print(f"Error: {result['error']}")